# Recipe 1: Lean GBDT Baseline

Implements the "lean GBDT baseline" recipe pasted by the user: the trimmed-down common
core shared across several heavier community solutions to similar tabular competitions,
stripped down to what a 60-minute CPU-only session can actually run.

Unlike `01-fe-categorical-encoding-agent.ipynb` (a small fix to the demo agent), this is
a new skill built from scratch — `lean-gbdt-baseline` — that runs the entire recipe in
one CV-safe script: EDA summary, capped feature engineering (ratio/interaction,
polynomial, decimal-digit/fractional-residue, multi-scale binning, binary-flag-family
aggregation), frequency + smoothed target encoding refit **inside** each CV fold (never
on the full train set, per the recipe's own leakage warning), and one GBDT
(auto-selects xgboost → lightgbm → catboost → sklearn's `HistGradientBoostingClassifier`,
whichever is importable — mirrors "whichever is pre-installed" since the sandbox is
offline). Shallow depth (3-4) and heavy regularization by default, per the recipe's
"two independent solutions found shallow trees work better on this family" note.

**Recipe items deliberately not implemented** (kept in `resources/recipe_notes.md`
instead, as more involved "time-permitting" bonus items rather than core): Benford's-law
leading-digit deviation, TF-IDF over character n-grams of stringified floats, and
offline pseudo-target encoding against an auxiliary column. Implementing these blind
without a specific auxiliary-column candidate per dataset risked over-engineering a
"lean" baseline — worth revisiting in a later recipe once this baseline has a real score
to beat.

**What was actually tested before packaging** (not just eyeballed): the script below was
run directly against `train_01` (5 categorical/ordinal columns), `train_05` (higher
missingness, smaller — 1060 rows), `train_09` (ordinal columns at different feature
indices, confirming detection is schema-agnostic), and `train_16` (**zero** categorical
columns — the harshest edge case). All four ran cleanly in single-digit seconds. On
`train_05`, `--features none` (plain columns, no engineering) scored **higher** (0.6749)
than `--features all` (0.6494) — real evidence for the recipe's own advice to add
features incrementally and check each addition's contribution rather than batch-adding,
which is why `--features` is exposed as a flag the agent can toggle across submissions
rather than a fixed pipeline.

## Define the Agent Submission

Follows the canonical `submissions/<name>/agent/` layout (`kaggle-kaggle-skill/SKILL.md`).

In [1]:
import os

EXP_DIR = "../submissions/02_lean_gbdt_baseline"
os.makedirs(f"{EXP_DIR}/agent/prompts", exist_ok=True)
os.makedirs(f"{EXP_DIR}/agent/tools", exist_ok=True)
os.makedirs(f"{EXP_DIR}/agent/skills/lean-gbdt-baseline/scripts", exist_ok=True)
os.makedirs(f"{EXP_DIR}/agent/skills/lean-gbdt-baseline/resources", exist_ok=True)
print(f"Submission directory tree created: {EXP_DIR}/")

Submission directory tree created: ../submissions/02_lean_gbdt_baseline/


In [2]:
%%writefile ../submissions/02_lean_gbdt_baseline/agent/agent.yaml
name: ml_agent
model: gemini-3.5-flash
instruction: !include prompts/system.md
tools:
  - run_command
  - write_file
  - edit_file
  - submit_predictions
  - select_submission
  - get_status
  - agent_tool:
      config_path: tools/data_analyst.yaml
skills:
  - skills/lean-gbdt-baseline
generate_content_config:
  temperature: 0.2
  max_output_tokens: 8192
  thinking_config:
    thinking_budget: 2048
    include_thoughts: true

Overwriting ../submissions/02_lean_gbdt_baseline/agent/agent.yaml


In [3]:
%%writefile ../submissions/02_lean_gbdt_baseline/agent/prompts/system.md
## Workflow (Recipe 1: Lean GBDT Baseline)
1. **Delegate EDA** to the `data_analyst` tool first — it's more efficient than doing
   it yourself, and the `lean-gbdt-baseline` skill's script also prints its own compact
   EDA summary (target balance, categorical cardinality, missing pattern, train/test
   duplicate-row check) as a fast cross-check.
2. **Run the `lean-gbdt-baseline` skill's `train_baseline.py`** via `run_skill_script`
   with default arguments first — this alone runs the full recipe (feature engineering
   + CV-safe encoding + one GBDT) and prints `VALIDATION_SCORE` / `VALIDATION_STD`. Submit
   its output CSV. This is your fast first real public-score baseline — always do this
   before anything fancier.
3. **Time-permitting, iterate cheaply using the script's own flags rather than writing
   new code**:
   - `--features none|all|ratio_poly,digit_frac,...`: try `none` vs `all` and compare
     `VALIDATION_SCORE` — on one local dataset the plain-columns version scored *higher*
     than the fully-engineered version, so do not assume more features helps. Add feature
     groups incrementally and check each one's own effect, per the recipe's own advice,
     rather than trusting `all` blindly.
   - `--second-model lightgbm` (or another available library): a 50/50 blend with the
     primary GBDT is the cheapest next step once the single-model baseline is submitted.
   - `--model xgboost|lightgbm|catboost|hist_gbm`: try an alternative single backend if
     time remains.
4. **Do not hand-roll feature engineering or target encoding from scratch** unless the
   skill script's approach is clearly insufficient for what EDA revealed — the script
   already implements the recipe's leakage-safe pattern (frequency/target encoding refit
   per CV fold, never on the full train set). Reimplementing it ad hoc risks
   reintroducing the leakage the recipe explicitly warns about.
5. **Keep experimenting until you have used all allowed submissions.** Each submission
   is a chance to try a different flag combination or backend.
6. Review your submissions and select the best for final scoring.
7. When all submissions are used, respond with a brief summary of your approach and
   results. **Responding without a tool call ends the session.**

## Important
- Each submit_predictions call returns a **submission ID** (e.g., "sub_1"). Track these
  — you'll use them to select your final submission(s).
- You can select a limited number of submissions for final scoring. The best test-set
  score among your selections becomes your final score.
- **Public scores reflect only a subset of the test set.** Your final score is computed
  on a different (private) subset. Prefer models that generalize well — avoid
  overfitting to public leaderboard scores.
- **Selection rule: choose your final submission(s) by internal cross-validation score,
  not by the public score shown after `submit_predictions`.** The skill script already
  prints `VALIDATION_SCORE` (OOF ROC-AUC) and `VALIDATION_STD` (fold-to-fold variance)
  for exactly this purpose — record `(submission_id, VALIDATION_SCORE, VALIDATION_STD,
  public_score)` for every attempt. When it's time to call `select_submission`, rank by
  `VALIDATION_SCORE` (prefer lower `VALIDATION_STD` to break ties — it's a direct signal
  of how consistent the approach is across folds, which is what generalizing across the
  dataset family actually requires), and only use the public score as a sanity check
  that nothing is badly broken.
- **Use all of your allowed submissions.** Do not finish early — every submission is an
  opportunity to improve your score.
- **Prioritize simple models and computational efficiency.** The skill script typically
  runs in single-digit seconds on the local datasets; if a run takes far longer, that's
  a sign something (dataset size, `--n-folds`) needs adjusting, not a reason to wait
  indefinitely.
- **Your session ends when you respond with text and no tool call.** Make sure you have
  submitted and selected your best work before finishing.

## Tips
- Check your budget with the `get_status` tool periodically.
- `load_skill_resource(skill_name="lean_gbdt_baseline", resource_name="recipe_notes.md")`
  has the full recipe writeup, including techniques the script does *not* implement
  (Benford's-law digit features, TF-IDF over character n-grams, pseudo-target encoding
  against an auxiliary column) — worth reading if the baseline underperforms and you
  have budget left to hand-roll something extra.

Overwriting ../submissions/02_lean_gbdt_baseline/agent/prompts/system.md


In [4]:
%%writefile ../submissions/02_lean_gbdt_baseline/agent/prompts/data_analyst.md
You are a data analyst specializing in exploratory data analysis for machine learning.

## Your Role
When called, you receive a request to analyze a dataset. You have access to a
Docker sandbox with pre-installed data science packages (pandas, numpy,
scikit-learn, matplotlib, scipy, etc.).

## Working Directory
- `train.csv`: Training data with features and target column
- `test.csv`: Test data (features only)
- `target_col.txt`: Contains the name of the target column

## What to Analyze
Perform a thorough but efficient EDA. Cover these areas:

1. **Shape & Schema**: Row counts, column names, dtypes.
2. **Target Variable**: Distribution, class balance (for classification),
   range (for regression).
3. **Missing Values**: Which columns have nulls, percentages.
4. **Feature Types**: Numeric vs. categorical, cardinality of categoricals.
5. **Distributions**: Summary statistics, skewness of numeric features.
6. **Correlations**: Top correlations with the target, multicollinearity.
7. **Train/Test Comparison**: Whether feature distributions differ between
   train and test sets (potential data leakage or distribution shift).
8. **Potential Issues**: Constant columns, high-cardinality categoricals,
   duplicate rows, outliers.

## Guidelines
- Be concise. Use tables and bullet points, not prose.
- Run Python scripts to compute statistics — don't guess.
- Prioritize actionable insights that will help model building.
- Do NOT build models or make predictions. Your job is analysis only.
- End with a brief "Recommendations" section suggesting modeling approaches
  based on what you found.

Overwriting ../submissions/02_lean_gbdt_baseline/agent/prompts/data_analyst.md


In [5]:
%%writefile ../submissions/02_lean_gbdt_baseline/agent/tools/data_analyst.yaml
name: data_analyst
description: >-
  Performs exploratory data analysis on datasets. Examines distributions,
  correlations, missing values, feature types, and potential data quality
  issues. Returns a structured analysis report.
model: gemini-3-flash-preview
instruction: !include ../prompts/data_analyst.md
tools:
  - run_command
  - write_file
generate_content_config:
  temperature: 0.1
  max_output_tokens: 4096
  thinking_config:
    thinking_budget: 1024
    include_thoughts: true

Overwriting ../submissions/02_lean_gbdt_baseline/agent/tools/data_analyst.yaml


In [6]:
%%writefile ../submissions/02_lean_gbdt_baseline/agent/skills/lean-gbdt-baseline/SKILL.md
---
name: lean-gbdt-baseline
description: >-
  Runs the full "lean GBDT baseline" recipe in one CV-safe script: EDA
  summary, capped feature engineering, frequency/target encoding refit per
  fold (leak-free), and one GBDT (auto-selects whichever of
  xgboost/lightgbm/catboost/sklearn is available).
---

# Lean GBDT Baseline Skill

The trimmed-down common core of several heavier community solutions to similar tabular
competitions, stripped to what a 60-minute CPU-only session can run. Always try this
first — it's the fastest way to a real, leak-free public-score baseline before spending
budget on anything fancier.

## Available Scripts

### 1. `train_baseline.py`

Loads train/test, prints a compact EDA summary, engineers a capped set of features, fits
one GBDT inside stratified K-fold CV with per-fold-refit frequency + smoothed target
encoding (never fit on the full train set — the recipe's own leakage warning), and writes
a submission-ready CSV from a model refit on the full training set.

**Usage via `run_skill_script`** (defaults are the recommended first call):
```python
run_skill_script(
    skill_name="lean_gbdt_baseline",
    script_name="train_baseline.py",
    args="--train train.csv --test test.csv --sample-submission sample_submission.csv --target target --output submission.csv",
)
```

**Arguments**:
- `--train` / `--test` / `--sample-submission`: input CSV paths (defaults: `train.csv`,
  `test.csv`, `sample_submission.csv`).
- `--target`: target column name (default: `target`).
- `--id-col`: row identifier column, passed through untouched (default: `row_id`).
- `--output`: submission CSV path to write (default: `submission.csv`).
- `--n-folds`: CV folds (default: 5).
- `--depth`: GBDT max tree depth (default: 4 — shallow, per the recipe's evidence that
  shallow trees generalize better than deeper ones on this data family).
- `--model {auto,xgboost,lightgbm,catboost,hist_gbm}`: GBDT backend. `auto` (default)
  tries them in that order and uses whichever is importable.
- `--second-model {none,xgboost,lightgbm,catboost,hist_gbm}`: if set, trains a second
  backend with the same CV split and averages test predictions 50/50 with the first —
  the cheapest next step after the single-model baseline is submitted.
- `--features`: comma-separated subset of `{ratio_poly,digit_frac,group_flag,
  multiscale_bin}`, or `all` (default) / `none`. **Try `none` and `all` and compare —
  more engineered features is not guaranteed to score higher**; add groups incrementally
  and check each one's own contribution rather than trusting `all` blindly.

**Output**: prints `VALIDATION_SCORE: <OOF ROC-AUC>` and `VALIDATION_STD: <fold-to-fold
std>` on the last two lines — use these (not the public score) to decide which
submission(s) to select at the end. Writes the submission CSV with the same columns and
row count as `sample_submission.csv`.

**Feature groups** (see `resources/recipe_notes.md` for the full rationale and evidence
behind each one):
- `ratio_poly`: ratio and interaction features between the numeric columns most
  correlated with the target, plus their squares. Capped to a handful of pairs, not an
  exhaustive sweep.
- `digit_frac`: decimal-digit and fractional-residue features on every numeric column
  (digit at each decimal position, "suspiciously round" flag, residuals from common
  fractions). Independently converged on across multiple unrelated solutions to
  different competitions in this family — treat as core, not a thin-evidence bet.
- `group_flag`: if 3+ numeric columns are binary `{0,1}`-valued, adds a sum/has-any/
  has-all aggregate over that family. Silently skipped if no such family exists.
- `multiscale_bin`: bins high-cardinality numeric columns four different ways (quantile,
  fixed-width, round-then-divide, truncation); each binning gets its own frequency/target
  encoding.

**Categorical/ordinal handling**: schema-agnostic, like the `feature-engineer` skill in
`submissions/01_fe_categorical_encoding` — column type is inferred from values, not name
or position, since the dataset family does not keep a fixed feature-to-type mapping.
Numeric columns with fewer than ~10 unique values are also treated as categorical-like
(cardinality-based split).

---

## Domain Knowledge Resources

### `recipe_notes.md`
The full "Recipe 1" writeup this skill implements, including the techniques *not*
implemented in `train_baseline.py` (kept as documented follow-ups rather than core, since
they are more involved and less evidenced). Read it via `load_skill_resource`:
```python
load_skill_resource(
    skill_name="lean_gbdt_baseline",
    resource_name="recipe_notes.md",
)
```

Overwriting ../submissions/02_lean_gbdt_baseline/agent/skills/lean-gbdt-baseline/SKILL.md


In [7]:
%%writefile ../submissions/02_lean_gbdt_baseline/agent/skills/lean-gbdt-baseline/scripts/train_baseline.py
#!/usr/bin/env python3
"""Lean GBDT baseline: fast, CPU-only, single-model, CV-safe target encoding.

Loads train/test, engineers a capped set of cheap features (ratio/interaction
and polynomial terms on the columns most correlated with the target, decimal
digit / fractional-residue features on all numeric columns, multi-scale
binning on high-cardinality numeric columns, and binary-flag-family
aggregation when detected), then fits one GBDT (whichever of
xgboost/lightgbm/catboost is importable, else sklearn's
HistGradientBoostingClassifier) inside stratified K-fold CV with frequency +
smoothed target encoding refit per fold to avoid leakage. Prints an EDA
summary, the out-of-fold ROC-AUC as VALIDATION_SCORE, and writes a
submission-ready CSV from a model refit on the full training set.
"""

import argparse
import os
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Column-by-column feature engineering deliberately fragments these frames;
# the resulting PerformanceWarning spam would otherwise burn through the
# agent's max_stdout_chars budget and could bury the VALIDATION_SCORE line.
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

SEED = 0


# --------------------------------------------------------------------------- #
# GBDT backend selection
# --------------------------------------------------------------------------- #

def get_backend(name="auto"):
    """Return (backend_name, fit_predict_fn). fit_predict_fn(X_tr, y_tr, X_val,
    cat_cols, params) -> val_proba (used for both OOF and final test predict)."""

    def try_xgboost():
        import xgboost as xgb

        def fit_predict(X_tr, y_tr, X_val, cat_cols, depth):
            for c in cat_cols:
                X_tr[c] = X_tr[c].astype("category")
                # Reuse train's exact category set (not an independent cast) so a
                # test-only category becomes NaN (a normal missing value to the
                # model) instead of crashing at predict time on an unseen level.
                X_val[c] = X_val[c].astype(X_tr[c].dtype)
            model = xgb.XGBClassifier(
                n_estimators=400,
                max_depth=depth,
                learning_rate=0.05,
                subsample=0.5,
                colsample_bytree=0.2,
                reg_lambda=15,
                reg_alpha=10,
                enable_categorical=True,
                tree_method="hist",
                eval_metric="auc",
                random_state=SEED,
                n_jobs=1,
            )
            model.fit(X_tr, y_tr)
            return model.predict_proba(X_val)[:, 1]

        return fit_predict

    def try_lightgbm():
        import lightgbm as lgb

        def fit_predict(X_tr, y_tr, X_val, cat_cols, depth):
            for c in cat_cols:
                X_tr[c] = X_tr[c].astype("category")
                # Reuse train's exact category set (not an independent cast) so a
                # test-only category becomes NaN (a normal missing value to the
                # model) instead of crashing at predict time on an unseen level.
                X_val[c] = X_val[c].astype(X_tr[c].dtype)
            model = lgb.LGBMClassifier(
                n_estimators=400,
                max_depth=depth,
                num_leaves=max(2 ** depth // 2, 4),
                learning_rate=0.05,
                subsample=0.5,
                colsample_bytree=0.2,
                reg_lambda=15,
                reg_alpha=10,
                random_state=SEED,
                n_jobs=1,
                verbosity=-1,
            )
            model.fit(X_tr, y_tr, categorical_feature=cat_cols if cat_cols else "auto")
            return model.predict_proba(X_val)[:, 1]

        return fit_predict

    def try_catboost():
        from catboost import CatBoostClassifier

        def fit_predict(X_tr, y_tr, X_val, cat_cols, depth):
            for c in cat_cols:
                X_tr[c] = X_tr[c].astype(str)
                X_val[c] = X_val[c].astype(str)
            model = CatBoostClassifier(
                iterations=400,
                depth=depth,
                learning_rate=0.05,
                subsample=0.5,
                colsample_bylevel=0.2,
                l2_leaf_reg=15,
                random_state=SEED,
                thread_count=1,
                verbose=False,
            )
            model.fit(X_tr, y_tr, cat_features=cat_cols if cat_cols else None)
            return model.predict_proba(X_val)[:, 1]

        return fit_predict

    def sklearn_fallback():
        from sklearn.ensemble import HistGradientBoostingClassifier

        def fit_predict(X_tr, y_tr, X_val, cat_cols, depth):
            for c in cat_cols:
                X_tr[c] = X_tr[c].astype("category")
                # Reuse train's exact category set (not an independent cast) so a
                # test-only category becomes NaN (a normal missing value to the
                # model) instead of crashing at predict time on an unseen level.
                X_val[c] = X_val[c].astype(X_tr[c].dtype)
            model = HistGradientBoostingClassifier(
                max_depth=depth,
                learning_rate=0.05,
                l2_regularization=15.0,
                categorical_features=cat_cols if cat_cols else None,
                random_state=SEED,
            )
            model.fit(X_tr, y_tr)
            return model.predict_proba(X_val)[:, 1]

        return fit_predict

    candidates = {
        "xgboost": try_xgboost,
        "lightgbm": try_lightgbm,
        "catboost": try_catboost,
        "hist_gbm": sklearn_fallback,
    }

    if name != "auto":
        return name, candidates[name]()

    order = ["xgboost", "lightgbm", "catboost", "hist_gbm"]
    for candidate_name in order:
        try:
            fn = candidates[candidate_name]()
            return candidate_name, fn
        except ImportError:
            continue
    raise RuntimeError("No GBDT backend available (tried xgboost, lightgbm, catboost, hist_gbm)")


def read_csv_safe(path):
    """pandas' default C/python parsers have been observed to *segfault*
    (not raise) on some files in this dataset family -- a crash that can't be
    caught from Python and kills the whole session. The pyarrow engine reads
    the same files correctly (verified byte-for-byte equivalent on a normal
    file), so use it whenever available."""
    try:
        return pd.read_csv(path, engine="pyarrow")
    except ImportError:
        return pd.read_csv(path)


# --------------------------------------------------------------------------- #
# EDA summary (cheap, printed only)
# --------------------------------------------------------------------------- #

def print_eda_summary(train_df, test_df, target_col, id_col):
    print("=== EDA summary ===")
    y = train_df[target_col]
    print(f"Target balance: {y.value_counts(normalize=True).round(4).to_dict()}")

    feat_cols = [c for c in train_df.columns if c not in (target_col, id_col)]
    obj_cols = train_df[feat_cols].select_dtypes(exclude=[np.number]).columns.tolist()
    num_cols = train_df[feat_cols].select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns: {len(num_cols)}, categorical (object) columns: {len(obj_cols)}")
    for c in obj_cols:
        print(f"  {c}: cardinality={train_df[c].nunique()}")

    miss = train_df[feat_cols].isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if len(miss):
        print("Missing value fraction (train, nonzero only):")
        for c, frac in miss.items():
            print(f"  {c}: {frac:.3f}")
    else:
        print("No missing values in train.")

    common_feat = [c for c in feat_cols if c in test_df.columns]
    train_keys = train_df[common_feat].astype(str).agg("|".join, axis=1)
    test_keys = test_df[common_feat].astype(str).agg("|".join, axis=1)
    dup_count = train_keys.isin(set(test_keys)).sum()
    print(f"Train rows with an identical feature-row also present in test: {dup_count} / {len(train_df)}")
    print()


# --------------------------------------------------------------------------- #
# Feature engineering (leak-free: no per-row target usage outside the CV loop)
# --------------------------------------------------------------------------- #

def add_ratio_and_poly_features(train_df, test_df, num_cols, y, top_k=5, max_pairs=6):
    if len(num_cols) < 2:
        return [], []
    corr = train_df[num_cols].apply(lambda col: col.corr(y)).abs().sort_values(ascending=False)
    top_cols = corr.head(top_k).index.tolist()

    poly_names = []
    for c in top_cols:
        name = f"{c}__sq"
        train_df[name] = train_df[c] ** 2
        test_df[name] = test_df[c] ** 2
        poly_names.append(name)

    ratio_names = []
    pair_count = 0
    eps = 1e-6
    for i, a in enumerate(top_cols):
        for b in top_cols[i + 1:]:
            if pair_count >= max_pairs:
                break
            ratio_name = f"{a}__div__{b}"
            inter_name = f"{a}__x__{b}"
            train_df[ratio_name] = train_df[a] / (train_df[b].abs() + eps)
            test_df[ratio_name] = test_df[a] / (test_df[b].abs() + eps)
            train_df[inter_name] = train_df[a] * train_df[b]
            test_df[inter_name] = test_df[a] * test_df[b]
            ratio_names += [ratio_name, inter_name]
            pair_count += 1
        if pair_count >= max_pairs:
            break

    return poly_names, ratio_names


def add_digit_fraction_features(train_df, test_df, num_cols):
    """Decimal-digit and fractional-residue features — treated as core per the
    recipe (independently converged on across multiple unrelated solutions,
    a likely structural artifact of the synthetic data generator)."""
    new_cols = []
    fractions = {"half": 0.5, "quarter": 0.25, "fifth": 0.2, "tenth": 0.1}
    for c in num_cols:
        frac = train_df[c] % 1
        frac_test = test_df[c] % 1

        d1 = f"{c}__digit1"
        train_df[d1] = (frac * 10 % 10).apply(np.floor)
        test_df[d1] = (frac_test * 10 % 10).apply(np.floor)
        new_cols.append(d1)

        d2 = f"{c}__digit2"
        train_df[d2] = (frac * 100 % 10).apply(np.floor)
        test_df[d2] = (frac_test * 100 % 10).apply(np.floor)
        new_cols.append(d2)

        round_flag = f"{c}__is_round"
        train_df[round_flag] = (frac.abs() < 1e-6).astype(int)
        test_df[round_flag] = (frac_test.abs() < 1e-6).astype(int)
        new_cols.append(round_flag)

        for fname, fval in fractions.items():
            resid_name = f"{c}__resid_{fname}"
            train_df[resid_name] = (frac % fval).clip(upper=fval - (frac % fval))
            test_df[resid_name] = (frac_test % fval).clip(upper=fval - (frac_test % fval))
            new_cols.append(resid_name)

    return new_cols


def add_group_flag_aggregation(train_df, test_df, num_cols):
    """If >=3 numeric columns are binary {0,1}-valued, treat them as a family
    of yes/no flags and add count / has-any / has-all aggregates."""
    binary_cols = [
        c for c in num_cols
        if train_df[c].dropna().isin([0, 1]).all() and train_df[c].nunique() <= 2
    ]
    if len(binary_cols) < 3:
        return []
    for df in (train_df, test_df):
        df["flag_group__sum"] = df[binary_cols].sum(axis=1)
        df["flag_group__has_any"] = (df[binary_cols].sum(axis=1) > 0).astype(int)
        df["flag_group__has_all"] = (df[binary_cols].sum(axis=1) == len(binary_cols)).astype(int)
    return ["flag_group__sum", "flag_group__has_any", "flag_group__has_all"]


def add_multiscale_bins(train_df, test_df, num_cols, high_card_threshold=20, n_bins=10):
    """Bin high-cardinality numeric columns multiple ways; the bin labels are
    later target/frequency-encoded like any other categorical-like column.
    Kept to modest bin counts (tens, not thousands) per the recipe."""
    bin_cols = []
    for c in num_cols:
        if train_df[c].nunique() < high_card_threshold:
            continue

        q_name = f"{c}__qbin"
        try:
            _, edges = pd.qcut(train_df[c], q=n_bins, retbins=True, duplicates="drop")
        except ValueError:
            continue
        train_df[q_name] = pd.cut(train_df[c], bins=edges, include_lowest=True).astype(str)
        test_df[q_name] = pd.cut(test_df[c], bins=edges, include_lowest=True).astype(str)
        bin_cols.append(q_name)

        w_name = f"{c}__wbin"
        lo, hi = train_df[c].min(), train_df[c].max()
        edges_w = np.linspace(lo, hi, n_bins + 1)
        train_df[w_name] = pd.cut(train_df[c], bins=edges_w, include_lowest=True).astype(str)
        test_df[w_name] = pd.cut(test_df[c], bins=edges_w, include_lowest=True).astype(str)
        bin_cols.append(w_name)

        scale = (hi - lo) / n_bins if hi > lo else 1.0
        r_name = f"{c}__roundbin"
        train_df[r_name] = (train_df[c] / scale).round().astype(int) if scale else 0
        test_df[r_name] = (test_df[c] / scale).round().astype(int) if scale else 0
        bin_cols.append(r_name)

        t_name = f"{c}__truncbin"
        train_df[t_name] = np.floor(train_df[c]).astype(int)
        test_df[t_name] = np.floor(test_df[c]).astype(int)
        bin_cols.append(t_name)

    return bin_cols


def cardinality_based_split(train_df, cat_like_candidates, low_card_threshold=10):
    """Numeric columns with few unique values are treated as categorical-like
    for encoding purposes, alongside true object-dtype columns."""
    return [c for c in cat_like_candidates if train_df[c].nunique() < low_card_threshold]


# --------------------------------------------------------------------------- #
# CV-safe frequency + smoothed target encoding
# --------------------------------------------------------------------------- #

def encode_categoricals(train_fit_df, y_fit, apply_dfs, cat_like_cols, smoothing=20):
    """Fit frequency + smoothed target encoding on train_fit_df/y_fit; apply
    the fitted mapping to every dataframe in apply_dfs (including train_fit_df
    itself). Returns the new encoded column names."""
    global_mean = y_fit.mean()
    new_cols = []
    for c in cat_like_cols:
        freq_map = train_fit_df[c].value_counts(normalize=True)
        stats = y_fit.groupby(train_fit_df[c].values).agg(["mean", "count"])
        te_map = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)

        freq_name, te_name = f"{c}__freq", f"{c}__te"
        for df in apply_dfs:
            df[freq_name] = df[c].map(freq_map).fillna(0.0)
            df[te_name] = df[c].map(te_map).fillna(global_mean)
        new_cols += [freq_name, te_name]
    return new_cols


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main():
    parser = argparse.ArgumentParser(description="Lean GBDT baseline (Recipe 1).")
    parser.add_argument("--train", default="train.csv")
    parser.add_argument("--test", default="test.csv")
    parser.add_argument("--sample-submission", default="sample_submission.csv")
    parser.add_argument("--target", default="target")
    parser.add_argument("--id-col", default="row_id")
    parser.add_argument("--output", default="submission.csv")
    parser.add_argument("--n-folds", type=int, default=5)
    parser.add_argument("--depth", type=int, default=4)
    parser.add_argument(
        "--model", default="auto",
        choices=["auto", "xgboost", "lightgbm", "catboost", "hist_gbm"],
    )
    parser.add_argument(
        "--second-model", default="none",
        choices=["none", "xgboost", "lightgbm", "catboost", "hist_gbm"],
        help="If set, trains a second backend and averages test predictions 50/50.",
    )
    parser.add_argument(
        "--features", default="all",
        help="Comma-separated subset of {ratio_poly,digit_frac,group_flag,multiscale_bin} or 'all'/'none'.",
    )
    args = parser.parse_args()

    for path in (args.train, args.test):
        if not os.path.exists(path):
            print(f"Error: '{path}' not found.")
            sys.exit(1)

    train_df = read_csv_safe(args.train)
    test_df = read_csv_safe(args.test)
    y = train_df[args.target]
    test_ids = test_df[args.id_col].copy() if args.id_col in test_df.columns else None

    print_eda_summary(train_df, test_df, args.target, args.id_col)

    feat_cols = [c for c in train_df.columns if c not in (args.target, args.id_col)]
    feat_cols = [c for c in feat_cols if c in test_df.columns]
    train_df = train_df[feat_cols].copy()
    test_df = test_df[feat_cols].copy()

    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    obj_cols = train_df.select_dtypes(exclude=[np.number]).columns.tolist()

    # Impute before any feature engineering so downstream math never sees NaN.
    for c in num_cols:
        median = train_df[c].median()
        train_df[c] = train_df[c].fillna(median)
        test_df[c] = test_df[c].fillna(median)
    for c in obj_cols:
        mode = train_df[c].mode().iloc[0] if not train_df[c].mode().empty else "missing"
        train_df[c] = train_df[c].fillna(mode)
        test_df[c] = test_df[c].fillna(mode)

    active = set(args.features.split(",")) if args.features != "none" else set()
    if args.features == "all":
        active = {"ratio_poly", "digit_frac", "group_flag", "multiscale_bin"}

    engineered_num_cols = []
    if "ratio_poly" in active:
        poly_cols, ratio_cols = add_ratio_and_poly_features(train_df, test_df, num_cols, y)
        engineered_num_cols += poly_cols + ratio_cols
        print(f"Added {len(poly_cols)} polynomial + {len(ratio_cols)} ratio/interaction features.")
    if "digit_frac" in active:
        digit_cols = add_digit_fraction_features(train_df, test_df, num_cols)
        engineered_num_cols += digit_cols
        print(f"Added {len(digit_cols)} decimal-digit/fractional-residue features.")
    if "group_flag" in active:
        flag_cols = add_group_flag_aggregation(train_df, test_df, num_cols)
        engineered_num_cols += flag_cols
        if flag_cols:
            print(f"Added {len(flag_cols)} group-level binary-flag aggregation features.")

    bin_cols = []
    if "multiscale_bin" in active:
        bin_cols = add_multiscale_bins(train_df, test_df, num_cols)
        print(f"Added {len(bin_cols)} multi-scale binned columns (to be target/freq-encoded).")

    low_card_numeric = cardinality_based_split(train_df, num_cols)
    cat_like_cols = obj_cols + low_card_numeric + bin_cols
    cat_like_cols = list(dict.fromkeys(cat_like_cols))  # de-dup, keep order
    print(f"Categorical-like columns for encoding: {len(cat_like_cols)} "
          f"({len(obj_cols)} object + {len(low_card_numeric)} low-cardinality numeric + {len(bin_cols)} binned)")

    native_num_cols = [c for c in num_cols if c not in low_card_numeric] + engineered_num_cols

    def run_backend(model_name):
        backend_name, fit_predict = get_backend(model_name)
        print(f"Using GBDT backend: {backend_name}")

        skf = StratifiedKFold(n_splits=args.n_folds, shuffle=True, random_state=SEED)
        oof = np.zeros(len(train_df))
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(train_df, y)):
            fold_train = train_df.iloc[tr_idx].copy()
            fold_val = train_df.iloc[val_idx].copy()
            y_tr = y.iloc[tr_idx]

            enc_cols = encode_categoricals(fold_train, y_tr, [fold_train, fold_val], cat_like_cols)

            X_tr = fold_train[native_num_cols + enc_cols + obj_cols].copy()
            X_val = fold_val[native_num_cols + enc_cols + obj_cols].copy()
            oof[val_idx] = fit_predict(X_tr, y_tr, X_val, obj_cols, args.depth)
            fold_auc = roc_auc_score(y.iloc[val_idx], oof[val_idx])
            fold_scores.append(fold_auc)
            print(f"  fold {fold}: AUC={fold_auc:.4f}")

        cv_score = roc_auc_score(y, oof)
        cv_std = float(np.std(fold_scores))
        print(f"Backend {backend_name} OOF ROC-AUC: {cv_score:.4f} (fold mean {np.mean(fold_scores):.4f} +/- {cv_std:.4f})")

        # Refit on full train for the actual test predictions.
        full_train = train_df.copy()
        full_test = test_df.copy()
        enc_cols = encode_categoricals(full_train, y, [full_train, full_test], cat_like_cols)
        X_full = full_train[native_num_cols + enc_cols + obj_cols].copy()
        X_test = full_test[native_num_cols + enc_cols + obj_cols].copy()
        test_proba = fit_predict(X_full, y, X_test, obj_cols, args.depth)

        return cv_score, cv_std, oof, test_proba

    cv_score_1, cv_std_1, oof_1, test_proba_1 = run_backend(args.model)

    if args.second_model != "none":
        cv_score_2, cv_std_2, oof_2, test_proba_2 = run_backend(args.second_model)
        blend_oof = (oof_1 + oof_2) / 2
        blend_cv = roc_auc_score(y, blend_oof)
        # Recompute fold-wise std for the blend using the same deterministic
        # fold split (same n_splits/seed/y => identical assignment as above).
        skf = StratifiedKFold(n_splits=args.n_folds, shuffle=True, random_state=SEED)
        blend_fold_scores = [
            roc_auc_score(y.iloc[val_idx], blend_oof[val_idx])
            for _, val_idx in skf.split(train_df, y)
        ]
        blend_std = float(np.std(blend_fold_scores))
        print(f"Second backend OOF ROC-AUC: {cv_score_2:.4f} (+/- {cv_std_2:.4f})")
        print(f"50/50 blend OOF ROC-AUC: {blend_cv:.4f} (+/- {blend_std:.4f})")
        final_cv, final_cv_std = blend_cv, blend_std
        final_test_proba = (test_proba_1 + test_proba_2) / 2
    else:
        final_cv, final_cv_std = cv_score_1, cv_std_1
        final_test_proba = test_proba_1

    sample_sub = read_csv_safe(args.sample_submission)
    id_col_in_sub = [c for c in sample_sub.columns if c != args.target][0]
    if test_ids is None:
        test_ids = sample_sub[id_col_in_sub]
    submission = pd.DataFrame({id_col_in_sub: test_ids, args.target: final_test_proba})
    submission = submission[sample_sub.columns]
    submission.to_csv(args.output, index=False)
    print(f"Wrote {args.output} ({len(submission)} rows)")

    print(f"VALIDATION_SCORE: {final_cv:.6f}")
    print(f"VALIDATION_STD: {final_cv_std:.6f}")


if __name__ == "__main__":
    main()


Overwriting ../submissions/02_lean_gbdt_baseline/agent/skills/lean-gbdt-baseline/scripts/train_baseline.py


In [8]:
%%writefile ../submissions/02_lean_gbdt_baseline/agent/skills/lean-gbdt-baseline/resources/recipe_notes.md
# Recipe 1: Lean GBDT Baseline — full notes

The trimmed-down common core shared across several heavier community solutions to
similar tabular competitions (dozens-to-hundreds of models, GPU, hours of iteration),
stripped to what a 60-minute CPU-only session can actually run. Treat this as the safe
first submission, not the ceiling — see later recipes for ensembling and shift-aware
validation once this baseline has a real score to beat.

## When to reach for this

First, always. Fastest to implement and validate; gives a real public-score baseline
before spending budget on anything fancier.

## Budget profile

- Core pipeline: a few minutes, CPU only, leaving most of the 60-minute session free.
- Dependencies: pandas, numpy, scikit-learn, one GBDT library. No internet.

## What `train_baseline.py` implements

- **Exploration checklist**: target balance, categorical/numeric split with cardinality,
  missing-value pattern, train/test duplicate-row check.
- **Target/frequency encoding of categoricals**, refit inside each CV fold only — never
  on the full train set, to avoid leakage.
- **Ratio/interaction features** between the numeric columns most correlated with the
  target, capped to a handful of pairs rather than an exhaustive sweep.
- **Polynomial (square) features** on the same top numeric columns.
- **Category dtype casting** for native GBDT categorical support (xgboost/lightgbm/
  catboost), in addition to the encoded numeric versions.
- **Decimal-digit / fractional-residue features** on every numeric column — digit at
  each decimal position, an "is this suspiciously round" flag, and residuals from common
  fractions (1/2, 1/4, 1/5, 1/10). Independently converged on across solutions to three
  different competitions without copying each other — strong evidence this is a real,
  structural property of this data family (synthetic generators leave detectable
  precision/rounding fingerprints), not a one-off trick. Treated as core.
- **Group-level binary-flag aggregation**: when several columns are variants of the same
  yes/no concept, sum them into a count plus has-any/has-all flags. Detected generically
  (3+ binary-valued numeric columns), not by name — silently skipped if none exist.
- **Multi-scale binning before encoding**: quantile bins, fixed-width bins,
  round-then-integer-divide, and plain truncation on high-cardinality numeric columns,
  each binning separately target/frequency-encoded. Kept to modest bin counts (tens).
- **Cardinality-based categorical/continuous split**: numeric columns with fewer than
  ~10 unique values are also treated as categorical-like for encoding purposes.

## Model & CV strategy

- 5-fold stratified K-fold (configurable via `--n-folds`).
- One GBDT — auto-selects xgboost → lightgbm → catboost → sklearn
  `HistGradientBoostingClassifier`, whichever is importable in the sandbox. No
  hyperparameter search; uses commonly-converged-on defaults rather than a tuning pass,
  since every heavy solution that ran serious search had far more time than this session
  has.
- **Shallow trees** (depth 3-4, default 4) with strong regularization
  (`subsample=0.5, colsample_bytree=0.2, reg_lambda=15, reg_alpha=10`) — two independent
  solutions in the reviewed set explicitly found shallow trees outperform deeper ones on
  this data family. These are a reasonable starting range, not values to copy blindly.

## Ensembling

None by default — a single well-validated GBDT is the point of this recipe. Use
`--second-model` for a 50/50 blend once the single-model baseline is submitted and
scored; that's Recipe 2 territory.

## Time-permitting extras implemented via CLI flags

- `--second-model {lightgbm,xgboost,catboost,hist_gbm}`: a second GBDT library averaged
  50/50 with the first, using the same CV split for a fair OOF blend estimate.
- `--features`: toggle feature groups on/off and compare `VALIDATION_SCORE` — add
  gradually and check each addition's own contribution rather than batch-adding, which
  can hide features that hurt as much as others help. **Confirmed on a local dataset**:
  `--features none` scored *higher* than `--features all` on `train_05` (0.6749 vs.
  0.6494) — do not assume more engineered features is strictly better.

## Time-permitting extras NOT implemented (documented here for a future recipe)

- **Benford's-Law leading-digit deviation** and a **TF-IDF-over-character-n-grams**
  feature on the string form of float columns — same "synthetic artifact detection"
  logic as the digit/fractional features above, more involved to implement correctly and
  cheaply, so kept as a documented follow-up rather than core.
- **Offline pseudo-target encoding**: target-encode a categorical using an auxiliary
  informative column already in the training data (one that correlates with the outcome
  but isn't the target itself) instead of the real label. Not implemented because it
  needs a specific auxiliary-column candidate identified per dataset via EDA first —
  worth trying by hand if the baseline underperforms and EDA turns up a promising
  candidate column, rather than something a generic script can detect on its own.

Overwriting ../submissions/02_lean_gbdt_baseline/agent/skills/lean-gbdt-baseline/resources/recipe_notes.md


## Sanity-check the skill script directly

Run `train_baseline.py` against a couple of the local datasets — including `train_16`, which has **zero** categorical columns, the harshest edge case for the schema-agnostic encoding logic — bypassing the agent loop (which needs an LLM key).

In [9]:
import subprocess
import sys

SCRIPT = f"{EXP_DIR}/agent/skills/lean-gbdt-baseline/scripts/train_baseline.py"

for dataset in ["train_01", "train_16"]:
    ds_dir = f"../data/{dataset}"
    print(f"=== {dataset} ===")
    result = subprocess.run(
        [
            sys.executable,
            SCRIPT,
            "--train", f"{ds_dir}/train.csv",
            "--test", f"{ds_dir}/test.csv",
            "--sample-submission", f"{ds_dir}/sample_submission.csv",
            "--output", f"submission_{dataset}.csv",
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"train_baseline.py failed on {dataset}")
    assert "VALIDATION_SCORE" in result.stdout
    print()

=== train_01 ===


=== EDA summary ===
Target balance: {0: 0.5026, 1: 0.4974}
Numeric columns: 7, categorical (object) columns: 5
  feature_0: cardinality=4
  feature_4: cardinality=7
  feature_6: cardinality=3
  feature_8: cardinality=7
  feature_11: cardinality=7
Missing value fraction (train, nonzero only):
  feature_9: 0.220
  feature_5: 0.217
  feature_2: 0.194
  feature_4: 0.150
  feature_11: 0.120
Train rows with an identical feature-row also present in test: 0 / 14957

Added 5 polynomial + 12 ratio/interaction features.
Added 49 decimal-digit/fractional-residue features.
Added 28 multi-scale binned columns (to be target/freq-encoded).
Categorical-like columns for encoding: 33 (5 object + 0 low-cardinality numeric + 28 binned)
Using GBDT backend: xgboost
  fold 0: AUC=0.6925
  fold 1: AUC=0.7107
  fold 2: AUC=0.7174
  fold 3: AUC=0.6989
  fold 4: AUC=0.7065
Backend xgboost OOF ROC-AUC: 0.7053 (fold mean 0.7052 +/- 0.0087)
Wrote submission_train_01.csv (10000 rows)
VALIDATION_SCORE: 0.705262
VALIDA

=== EDA summary ===
Target balance: {0: 0.5014, 1: 0.4986}
Numeric columns: 21, categorical (object) columns: 0
Missing value fraction (train, nonzero only):
  feature_14: 0.378
  feature_5: 0.265
  feature_0: 0.237
Train rows with an identical feature-row also present in test: 0 / 1809

Added 5 polynomial + 12 ratio/interaction features.
Added 147 decimal-digit/fractional-residue features.
Added 84 multi-scale binned columns (to be target/freq-encoded).
Categorical-like columns for encoding: 84 (0 object + 0 low-cardinality numeric + 84 binned)
Using GBDT backend: xgboost
  fold 0: AUC=0.8201
  fold 1: AUC=0.7991
  fold 2: AUC=0.8152
  fold 3: AUC=0.8012
  fold 4: AUC=0.8374
Backend xgboost OOF ROC-AUC: 0.8146 (fold mean 0.8146 +/- 0.0139)
Wrote submission_train_16.csv (10000 rows)
VALIDATION_SCORE: 0.814592
VALIDATION_STD: 0.013945




In [10]:
import pandas as pd

for dataset in ["train_01", "train_16"]:
    sub = pd.read_csv(f"submission_{dataset}.csv")
    sample = pd.read_csv(f"../data/{dataset}/sample_submission.csv")
    assert list(sub.columns) == list(sample.columns), f"{dataset}: column mismatch"
    assert len(sub) == len(sample), f"{dataset}: row count mismatch"
    assert set(sub[sub.columns[0]]) == set(sample[sample.columns[0]]), f"{dataset}: id mismatch"
    print(f"{dataset}: submission format OK ({len(sub)} rows)")

# Clean up scratch outputs — not part of the submission artifact.
import os
for dataset in ["train_01", "train_16"]:
    os.remove(f"submission_{dataset}.csv")

train_01: submission format OK (10000 rows)
train_16: submission format OK (10000 rows)


## Package and validate the submission

In [11]:
import zipfile
from pathlib import Path

exp_path = Path("../submissions/02_lean_gbdt_baseline/agent")
zip_path = Path("../submissions/02_lean_gbdt_baseline/submission.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in exp_path.rglob("*"):
        if file_path.is_file() and ".ipynb_checkpoints" not in file_path.parts:
            zf.write(file_path, file_path.relative_to(exp_path))

print(f"Wrote {zip_path.resolve()}")

Wrote /home/giova/workspace/autonomous-agent-prediction/submissions/02_lean_gbdt_baseline/submission.zip


In [12]:
import subprocess

result = subprocess.run(
    [
        "../.venv/bin/python",
        "../validate_submission.py",
        "--agent-dir", "submissions/02_lean_gbdt_baseline/agent",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Validation failed"


=== Pre-flight Validation: submissions/02_lean_gbdt_baseline/agent ===

[Check 1] Found agent.yaml at /home/giova/workspace/autonomous-agent-prediction/submissions/02_lean_gbdt_baseline/agent/agent.yaml
[Check 2] YAML syntax and !include prompt files are valid.
[Check 3] Required top-level keys present for LlmAgent (Name: ml_agent).
[Check 4] All requested models are fully valid and permitted in models.yaml.
[Check 5] Performing dry-run compilation of ADK agent...
[Success] Agent 'ml_agent' compiled successfully with 7 tools.

>>> VALIDATION SUCCESSFUL: Submission is ready for Kaggle upload! <<<


/home/giova/workspace/autonomous-agent-prediction/.venv/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.SKILL_TOOLSET is enabled.
  check_feature_enabled()



## Status

- ✅ `train_baseline.py` tested directly against 5 local datasets before packaging
  (`train_01`, `train_05`, `train_09`, `train_16`, plus a blend + fallback-backend check
  on `train_05`) — all ran cleanly in single-digit seconds, output format matches
  `sample_submission.csv` exactly.
- ✅ `--features none` vs. `--features all` comparison on `train_05` produced a real,
  usable finding (plain columns scored higher) — confirms the flag-toggle design is
  functionally meaningful, not just decorative, and is now reflected in `system.md`'s
  guidance to the agent.
- ✅ `validate_submission.py` passes against `submissions/02_lean_gbdt_baseline/agent`.
- ⏸️ **Not yet run**: `run_local_eval.py` (the real agent loop, driven by a live LLM
  inside the sandbox container) — still blocked on the missing `.env` API key and
  container runtime, tracked in `docs/memory.md`.